In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


In [10]:
# Load the dataset
data = pd.read_csv("C:\StudPerformancePred\Student_performance_data _.csv")  # Adjust path if needed

# Inspect the dataset
print("Dataset shape:", data.shape)
print(data.head())
print(data.info())


Dataset shape: (2392, 15)
   StudentID  Age  Gender  Ethnicity  ParentalEducation  StudyTimeWeekly  \
0       1001   17       1          0                  2        19.833723   
1       1002   18       0          0                  1        15.408756   
2       1003   15       0          2                  3         4.210570   
3       1004   17       1          0                  3        10.028829   
4       1005   17       1          0                  2         4.672495   

   Absences  Tutoring  ParentalSupport  Extracurricular  Sports  Music  \
0         7         1                2                0       0      1   
1         0         0                1                0       0      0   
2        26         0                2                0       0      0   
3        14         0                3                1       0      0   
4        17         1                3                0       0      0   

   Volunteering       GPA  GradeClass  
0             0  2.929196       

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\user\AppData\Local\Temp\ipykernel_10972\4070055148.py:2: SyntaxWarning: invalid escape sequence '\S'
  data = pd.read_csv("C:\StudPerformancePred\Student_performance_data _.csv")  # Adjust path if needed


In [11]:
# Define features and target
X = data.drop(columns=["GradeClass"])  # Replace "GradeClass" with the actual target column name
y = data["GradeClass"]

# Inspect features and target
print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (2392, 14)
Target shape: (2392,)


In [12]:
# Handle missing values in the dataset
X = X.fillna(X.median())


In [13]:
# Example: Add new features
if "GPA" in X.columns and "Attendance" in X.columns:
    X["GPA_Squared"] = X["GPA"] ** 2
    X["Attendance_GPA_Ratio"] = X["Attendance"] / (X["GPA"] + 1)

# Inspect the dataset after feature engineering
print(X.head())


   StudentID  Age  Gender  Ethnicity  ParentalEducation  StudyTimeWeekly  \
0       1001   17       1          0                  2        19.833723   
1       1002   18       0          0                  1        15.408756   
2       1003   15       0          2                  3         4.210570   
3       1004   17       1          0                  3        10.028829   
4       1005   17       1          0                  2         4.672495   

   Absences  Tutoring  ParentalSupport  Extracurricular  Sports  Music  \
0         7         1                2                0       0      1   
1         0         0                1                0       0      0   
2        26         0                2                0       0      0   
3        14         0                3                1       0      0   
4        17         1                3                0       0      0   

   Volunteering       GPA  
0             0  2.929196  
1             0  3.042915  
2             

In [14]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["number"]).columns

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


Categorical columns: Index([], dtype='object')
Numerical columns: Index(['StudentID', 'Age', 'Gender', 'Ethnicity', 'ParentalEducation',
       'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport',
       'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA'],
      dtype='object')


In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectFromModel(RandomForestClassifier(random_state=42), threshold='median')),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

In [ ]:
# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse=False), categorical_features)
    ])

In [ ]:
# Build the full pipeline
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("feature_selection", SelectFromModel(RandomForestClassifier(random_state=42), threshold="median")),
    ("classifier", RandomForestClassifier(random_state=42))
])


In [17]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)


Training set size: (1913, 14)
Testing set size: (479, 14)


In [ ]:
# Train the model
pipeline.fit(X_train, y_train)


In [ ]:
# Evaluate the model
accuracy = pipeline.score(X_test, y_test)

print(f"Model Accuracy after Feature Engineering: {accuracy:.2%}")
